<a href="https://colab.research.google.com/github/Lobnaait/SEARCH_Lobna_Tsetline_CMRI/blob/main/Tsetline_ImagesDataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install pyTsetlinMachine

In [5]:
#IMPORT LIBRARIES AND DATASET
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from pyTsetlinMachine.tm import MultiClassTsetlinMachine

dataset = load_digits()  # Load the images dataset

In [6]:
# Divide the dataset into training set and test set for features (X) and target (y)
from sklearn.model_selection import train_test_split
X = dataset.images # input images
y = dataset.target # output labels
X_train, X_test, y_train, y_test = train_test_split(dataset.images, dataset.target, test_size = 0.25)
X_train2, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.20)



# How many tresholds are better to use?
Since the treshold is an hyperparameter I have to perform the validation step to obtain the optiman treshold.
For validation we use part of the training data

In [7]:
# CHOOSE THE TRESHOLDS: Based on the number of treshold divide the dataset into tresholds
def compute_thresholds(X, n_thresholds):
  # Extract non-zero pixel
  non_zero_pixels = X[X > 0]
  percentile_values = np.linspace(0, 100, int(n_thresholds) + 2)[1:-1] # Exclude 0 and 100
  thresholds = np.percentile(non_zero_pixels, percentile_values)
  return np.unique(thresholds)

In [8]:
def booleanise(X, thresholds):
  # Flatten each image
  X_flat = X.reshape(X.shape[0], -1)
  #Compare every pixel with every threshold
  # X_flat has shape (num_samples, num_pixels)
  # X_flat[:, :, None] -> (num_samples, num_pixels, 1)
  # thresholds has shape (num_thresholds,) after compute_thresholds fix
  # thresholds[None, None, :] -> (1, 1, num_thresholds)
  # Broadcasting will result in (num_samples, num_pixels, num_thresholds)
  X_bool = (X_flat[:, :, None] > thresholds[None, None, :])
  #convert to binary
  X_bool = X_bool.astype(np.uint32)
  return X_bool

In [ ]:
# VALIDATION: Test how many tresholds are optimal to use?
from pyTsetlinMachine.tm import MultiClassTsetlinMachine
from sklearn.metrics import accuracy_score
import numpy as np

threshold_candidates = np.linspace(0,100,1)
results = []

for n_thresholds in threshold_candidates:

    thresholds = compute_thresholds(X_val, n_thresholds)
    X_train_bool = booleanise(X_train, thresholds)
    X_val_bool = booleanise(X_val, thresholds)

    # model
    tm = MultiClassTsetlinMachine(number_of_clauses=1000, T=50, s=5.0)
    tm.fit(X_train_bool, y_train, epochs=100)

    val_accuracy = accuracy_score(y_val, tm.predict(X_val_bool))

    results.append(n_thresholds; val_accuracy)